# Lab08.3 — Traffic-Sign Recognition Using a Binarized Neural Network on FPGA

## Objectives

In this laboratory, you will run a binarized neural network (BNN) accelerator in the programmable logic (PL) of the Zybo board. Python running on the ARM processor (PS) prepares the image, transfers it to the FPGA through AXI DMA, and reads the classification scores.

By the end of the experiment, you will be able to:

- Load an FPGA overlay from Python and identify the AXI DMA interface.
- Prepare a 32 × 32 RGB image for hardware inference.
- Transfer image data to the accelerator and interpret its output scores.
- Compare FPGA predictions with reference labels from the GTSRB dataset.
- Measure end-to-end inference time and identify the contribution of software and data transfers.
- Optionally classify an image captured with a Logitech C270 webcam.

## Hardware and data

- **Board:** Digilent Zybo Legacy (XC7Z010), PYNQ 3.0.1.
- **FPGA design:** `../overlays/compact_gtsrb.bit` with its matching `.hwh` file.
- **Input:** RGB `uint8` image of 32 × 32 pixels (3072 bytes).
- **Output:** 44 signed `int16` scores (88 bytes). Indices 0–42 represent GTSRB classes; index 43 is unused padding and must not be interpreted as a traffic-sign class.
- **Communication:** PS ↔ PL using AXI DMA and contiguous memory buffers.

The first code cell loads the FPGA design. The following sections test DMA communication, classify one labeled traffic sign, evaluate a set of 20 images, and optionally process a webcam frame.

**Required files:** `Lab08/overlays/compact_gtsrb.bit`, `Lab08/overlays/compact_gtsrb.hwh`, `Lab08/tools/gtsrb.py`, and `Lab08/notebooks/gtsrb_batch20_inputs.npz` (20 prepared GTSRB test images with labels). The 20-image test also requires `Lab08/notebooks/BATCH20_CELL.py`. Open this notebook from the `Lab08/notebooks/` directory.


## 1 — Load the implemented FPGA design

In [ ]:
from pathlib import Path
import time
import numpy as np
from pynq import Overlay, allocate

bitfile = Path('../overlays/compact_gtsrb.bit').resolve()
hwhfile = bitfile.with_suffix('.hwh')
if not bitfile.is_file() or not hwhfile.is_file():
    raise FileNotFoundError('Overlay files not found in Lab08/overlays: {} and {}'.format(bitfile, hwhfile))

ol = Overlay(str(bitfile))
if 'axi_dma_0' not in ol.ip_dict:
    raise RuntimeError('DMA axi_dma_0 was not found in the loaded HWH.')
dma = ol.axi_dma_0
print('Loaded FPGA:', bitfile.name)
print('AXI DMA base:', hex(ol.ip_dict['axi_dma_0']['phys_addr']))
print('MM2S 8-bit, S2MM 16-bit, expected frame: 3072 input bytes / 88 output bytes')
print('Initial DMA status: TX=0x{:08X}, RX=0x{:08X}'.format(
    dma.mmio.read(0x04), dma.mmio.read(0x34)))


## 2 — DMA inference function

Receive DMA starts before transmit. Timeout and status checks prevent an infinite wait if a stream transfer fails. The two DMA buffers are reused. After a timeout reload the overlay.

In [ ]:
# The PYNQ allocator returns physically contiguous, DMA-safe memory.
image_buffer = allocate(shape=(3072,), dtype=np.uint8)
score_buffer = allocate(shape=(44,), dtype=np.int16)

def infer_fpga(rgb32, timeout_s=8.0):
    image = np.asarray(rgb32)
    if image.shape != (32, 32, 3) or image.dtype != np.uint8:
        raise ValueError('Expected RGB image: shape=(32,32,3), dtype=uint8')

    t0 = time.perf_counter()
    image_buffer[:] = np.ascontiguousarray(image).reshape(-1)
    image_buffer.flush()
    score_buffer[:] = 0
    score_buffer.flush()

    # S2MM is armed first so the first score cannot be lost.
    dma.recvchannel.transfer(score_buffer)
    dma.sendchannel.transfer(image_buffer)
    deadline = time.monotonic() + timeout_s
    while not (dma.sendchannel.idle and dma.recvchannel.idle):
        if dma.sendchannel.error or dma.recvchannel.error:
            raise RuntimeError('DMA error: TX=0x{:08X}, RX=0x{:08X}'.format(
                dma.mmio.read(0x04), dma.mmio.read(0x34)))
        if time.monotonic() >= deadline:
            raise TimeoutError('DMA timeout: TX=0x{:08X}, RX=0x{:08X}. '
                'Reload overlay before retry; check TLAST/weights/clocks.'.format(
                    dma.mmio.read(0x04), dma.mmio.read(0x34)))

    # The bounded poll above makes both waits return immediately on success.
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    score_buffer.invalidate()
    scores = np.array(score_buffer, dtype=np.int16, copy=True)
    t_ms = (time.perf_counter() - t0) * 1000.0

    winner = int(np.argmax(scores))
    # Class 43 is the unused padding output, NOT a GTSRB traffic-sign class.
    label = winner if winner < 43 else 'INVALID_OUTPUT_43'
    top3 = np.argsort(scores[:43])[-3:][::-1].astype(int).tolist()
    return dict(label=label, raw_argmax=winner, scores=scores,
                top3=[(i, int(scores[i])) for i in top3], elapsed_ms=t_ms)

def show_result(result):
    print('FPGA prediction:', result['label'])
    print('Top 3 valid classes (raw integer scores):', result['top3'])
    print('Padding score (class 43):', int(result['scores'][43]))
    print('End-to-end DMA time: {:.2f} ms'.format(result['elapsed_ms']))
    print('Raw scores:', result['scores'].tolist())
    print('Scores are NOT probabilities.')


## 3 — First hardware transfer

This transfers an artificial test pattern to verify AXI DMA communication. It is not an accuracy test.

In [ ]:
rgb_test = np.zeros((32, 32, 3), dtype=np.uint8)
rgb_test[:, :16, 0] = 255  # red on left
rgb_test[:, 16:, 1] = 255  # green on right
result = infer_fpga(rgb_test)
show_result(result)
assert len(result['scores']) == 44
print('PASS: complete DMA transaction with 44 int16 scores')


## 4 — FPGA inference on one labeled GTSRB test image

The repository includes `gtsrb_batch20_inputs.npz`, containing 20 prepared GTSRB test images and their reference class IDs. Select one image using `SAMPLE_INDEX` (0–19), display it, run FPGA inference and compare the predicted class with the reference label.

The experiment uses local files only. **No internet connection or external dataset download is required.**

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()  # Notebook in Lab08/notebooks/
TOOLS = ROOT / 'tools'
if not (TOOLS / 'gtsrb.py').is_file():
    raise FileNotFoundError('Missing Lab08/tools/gtsrb.py: {}'.format(TOOLS / 'gtsrb.py'))
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
from gtsrb import LABELS

BATCH_FILE = Path('gtsrb_batch20_inputs.npz')
if not BATCH_FILE.is_file():
    raise FileNotFoundError(
        'Missing {}. Copy the complete Lab08/notebooks directory from GitHub.'
        .format(BATCH_FILE.resolve()))

with np.load(BATCH_FILE, allow_pickle=False) as data:
    rgb_images = np.asarray(data['rgb32'])
    true_classes = np.asarray(data['true_classes'])
    filenames = np.asarray(data['filenames'])

if rgb_images.shape != (20, 32, 32, 3) or rgb_images.dtype != np.uint8:
    raise ValueError('Expected 20 RGB uint8 images with shape (20,32,32,3).')
if true_classes.shape != (20,):
    raise ValueError('Expected 20 reference class IDs.')
if not np.all((true_classes >= 0) & (true_classes < len(LABELS))):
    raise ValueError('One or more reference class IDs are invalid.')

SAMPLE_INDEX = 4  # Choose any number from 0 to 19
if not 0 <= SAMPLE_INDEX < len(rgb_images):
    raise IndexError('SAMPLE_INDEX must be between 0 and 19.')

rgb32 = np.array(rgb_images[SAMPLE_INDEX], copy=True)
true_id = int(true_classes[SAMPLE_INDEX])
filename = str(filenames[SAMPLE_INDEX])

plt.figure(figsize=(4, 4))
plt.imshow(rgb32, interpolation='nearest')
plt.title('GTSRB test image (32x32 RGB)')
plt.axis('off')
plt.show()

print('Image:', filename)
print('Ground truth: {} — {}'.format(true_id, LABELS[true_id]))
result = infer_fpga(rgb32)
show_result(result)
pred_id = result['label']
if isinstance(pred_id, int):
    print('Prediction name: {} — {}'.format(pred_id, LABELS[pred_id]))
else:
    print('Prediction name: invalid output (padding class 43)')
print('CORRECT:', pred_id == true_id)


In [ ]:
# Save the selected input, ground-truth label and FPGA output for later analysis.
np.savez_compressed(
    'fpga_compare_single.npz',
    rgb32=rgb32,
    fpga_scores=result['scores'],
    true_class=true_id,
    sample_index=SAMPLE_INDEX
)
print('Saved: fpga_compare_single.npz')


## 5 — Evaluate 20 GTSRB test images

The file `gtsrb_batch20_inputs.npz` contains a prepared sample of 20 labeled traffic-sign images. Run the next cell to display predictions, count correct classifications and measure the mean end-to-end inference time.

The sample contains only 20 images, so its accuracy must not be interpreted as the accuracy of the model on the complete GTSRB test set.

In [ ]:
%run -i BATCH20_CELL.py


## 6 — Logitech C270: camera capture and FPGA inference

Run the following cell to define the camera acquisition function. The camera captures one frame, extracts the center square, resizes it to 32 × 32 RGB and passes it to the FPGA. The camera is released after each capture.

In the next section, use **Capture & classify** to repeat the experiment without rerunning the code cell.

In [ ]:
# Camera acquisition function used by the interactive experiment below.
import cv2
import numpy as np

def capture_camera_and_infer():
    cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError('Camera unavailable at /dev/video0.')

        cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        cap.set(cv2.CAP_PROP_FPS, 15)

        ok, frame = False, None
        for _ in range(3):
            ok, frame = cap.read()
            if ok and frame is not None:
                break
        if not ok or frame is None:
            raise RuntimeError('Camera opened but no image was captured.')

        h, w = frame.shape[:2]
        side = min(h, w)
        x0, y0 = (w - side) // 2, (h - side) // 2
        crop_bgr = frame[y0:y0+side, x0:x0+side]
        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        camera_rgb32 = cv2.resize(
            crop_rgb, (32, 32), interpolation=cv2.INTER_AREA)
        prediction = infer_fpga(camera_rgb32)
        return crop_rgb, prediction
    finally:
        cap.release()  # Release /dev/video0 even if capture or FPGA inference fails.

print('Camera capture function ready. Use Capture & classify in the next section.')


## 7 — Interactive camera experiment: select a sign and classify it

The left panel displays one of the **20 schematic GTSRB reference signs**. Use **Previous / Next** or the number selector to change the sign. Present the selected sign to the Logitech C270 (for example, by displaying it on a separate screen or phone), then click **Capture & classify**. The right panel shows the actual camera frame, the FPGA prediction and the elapsed time.

You can repeat the process as often as needed **without rerunning this cell**. The button acquires a new camera frame for each attempt. The FPGA does not receive the illustration directly: it classifies the frame captured by the camera. These illustrations are not the original GTSRB test photographs, so results can differ from the offline benchmark.

**Note:** The selected reference class is used only to report whether the camera prediction matches the sign you intended to show. The software does not verify that the correct sign is actually in front of the camera.

In [ ]:
# Interactive 20-sign viewer and repeatable FPGA camera inference.
from IPython.display import display, SVG, clear_output
import ipywidgets as widgets
import json
import io
from PIL import Image

DISPLAY_ROOT = Path('../display').resolve()
manifest_path = DISPLAY_ROOT / 'manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(
        'Missing Lab08/display/manifest.json. Download the complete Lab08 directory from GitHub.')
with manifest_path.open('r', encoding='utf-8') as f:
    signs = json.load(f)

sign_index = widgets.BoundedIntText(
    value=1, min=1, max=len(signs), description='Sign:',
    layout=widgets.Layout(width='160px'))
previous_button = widgets.Button(description='Previous', icon='arrow-left')
next_button = widgets.Button(description='Next', icon='arrow-right')
capture_button = widgets.Button(
    description='Capture & classify', button_style='success', icon='camera',
    layout=widgets.Layout(width='190px'))
reference_output = widgets.Output(layout=widgets.Layout(width='52%'))
camera_output = widgets.Output(layout=widgets.Layout(width='46%'))

def show_sign(change=None):
    sign = signs[sign_index.value - 1]
    image_path = DISPLAY_ROOT / sign['image']
    if not image_path.is_file():
        raise FileNotFoundError('Missing sign illustration: {}'.format(image_path))
    with reference_output:
        clear_output(wait=True)
        display(widgets.HTML('<b>Reference: {} (class {})</b>'.format(
            sign['label'], sign['class_id'])))
        display(widgets.HTML(
            '<div style="max-width:330px">{}<div>'.format(
                image_path.read_text(encoding='utf-8').replace(
                    '<svg ', '<svg style="width:100%;height:auto" ', 1))))

def change_sign(step):
    sign_index.value = ((sign_index.value - 1 + step) % len(signs)) + 1

def on_capture(_):
    capture_button.disabled = True
    sign = signs[sign_index.value - 1]
    with camera_output:
        clear_output(wait=True)
        print('Capturing camera frame and running FPGA inference...')
    try:
        crop_rgb, prediction = capture_camera_and_infer()
        # capture_camera_and_infer() has already released /dev/video0.
        pred_id = prediction['label']
        with camera_output:
            clear_output(wait=True)
            display(widgets.HTML('<b>Logitech C270 — captured frame</b>'))
            image = Image.fromarray(crop_rgb)
            image.thumbnail((320, 320))
            display(image)
            pred_name = LABELS[pred_id] if isinstance(pred_id, int) else str(pred_id)
            display(widgets.HTML(
                '<b>FPGA prediction: {} (class {})</b><p>Expected sign: {} (class {})</p>'
                '<p>Matches selected reference: <b>{}</b></p>'
                '<p>End-to-end inference: {:.2f} ms</p>'.format(
                    pred_name, pred_id, sign['label'], sign['class_id'],
                    'YES' if pred_id == sign['class_id'] else 'NO',
                    prediction['elapsed_ms'])))
    except Exception as exc:
        with camera_output:
            clear_output(wait=True)
            print('Camera or inference error:', exc)
    finally:
        capture_button.disabled = False

previous_button.on_click(lambda _: change_sign(-1))
next_button.on_click(lambda _: change_sign(1))
capture_button.on_click(on_capture)
sign_index.observe(show_sign, names='value')

display(widgets.HBox([previous_button, sign_index, next_button, capture_button]))
display(widgets.HBox([reference_output, camera_output]))
show_sign()


## 8 — End-to-end camera processing benchmark

This experiment measures a **sequence of camera frames** using the same FPGA inference function as above. It reports the time for each stage:

- **Capture:** obtaining a frame from the Logitech C270 via OpenCV.
- **Preprocessing:** center crop, resize to 32 × 32 and BGR-to-RGB conversion on ARM.
- **Inference:** Python buffer handling, AXI DMA and FPGA execution (not FPGA core latency alone).
- **Total:** complete processing time for each frame.

Run the cell after Sections 1 and 2 have initialized the overlay and `infer_fpga()`. **Section 7 releases the camera after each capture (including on error) using `finally: cap.release()`. No separate Release button or cleanup cell is required.** Do not press Capture & classify while the benchmark is running. Change `NUM_FRAMES` to repeat the measurement with a larger sample.

**Interpretation:** The measured processing throughput does not by itself establish motion-to-result latency. The camera is configured for a nominal 15 FPS, but buffering, exposure, driver behavior and USB transfer can affect the actual rate. The benchmark opens the camera once, measures consecutive frames and excludes device initialization from the per-frame timings. It does not include traffic-sign localization, communication with a vehicle or vehicle reaction time.

After the graph, an **automatically generated interpretation in English** uses the measurements from your own run rather than fixed example values.

In [ ]:
# Camera → ARM preprocessing → AXI DMA/FPGA benchmark.
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

NUM_FRAMES = 20
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
capture_ms, preprocess_ms, inference_ms, total_ms = [], [], [], []
predictions = []

try:
    if not cap.isOpened():
        raise RuntimeError('Camera unavailable at /dev/video0.')
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 15)

    # Discard initial frames before measurements.
    for _ in range(3):
        cap.read()

    test_start = time.perf_counter()
    for i in range(NUM_FRAMES):
        t0 = time.perf_counter()
        ok, frame = cap.read()
        t1 = time.perf_counter()
        if not ok or frame is None:
            raise RuntimeError('Failed to capture frame {}'.format(i + 1))

        h, w = frame.shape[:2]
        side = min(h, w)
        x0, y0 = (w - side) // 2, (h - side) // 2
        crop = frame[y0:y0+side, x0:x0+side]
        small = cv2.resize(crop, (32, 32), interpolation=cv2.INTER_AREA)
        rgb32 = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
        t2 = time.perf_counter()

        output = infer_fpga(rgb32)
        t3 = time.perf_counter()

        capture_ms.append((t1 - t0) * 1000)
        preprocess_ms.append((t2 - t1) * 1000)
        inference_ms.append((t3 - t2) * 1000)
        total_ms.append((t3 - t0) * 1000)
        predictions.append(output['label'])
    elapsed_s = time.perf_counter() - test_start
finally:
    cap.release()

print('Processed frames:', len(total_ms))
print('Mean capture:       {:.2f} ms'.format(np.mean(capture_ms)))
print('Mean preprocessing: {:.2f} ms'.format(np.mean(preprocess_ms)))
print('Mean FPGA call:     {:.2f} ms'.format(np.mean(inference_ms)))
print('Mean total/frame:   {:.2f} ms'.format(np.mean(total_ms)))
print('Observed throughput: {:.2f} frames/s'.format(len(total_ms) / elapsed_s))
print('Mean total-only equivalent: {:.2f} frames/s'.format(
    1000.0 / np.mean(total_ms)))
print('FPGA predictions:', predictions)
print('Note: these are processing measurements, not sign-detection accuracy.')

fig, ax = plt.subplots(figsize=(9, 3.8))
indices = np.arange(1, len(total_ms) + 1)
ax.plot(indices, capture_ms, label='Capture')
ax.plot(indices, preprocess_ms, label='Preprocessing')
ax.plot(indices, inference_ms, label='FPGA inference call')
ax.plot(indices, total_ms, label='Total')
ax.set_xlabel('Frame')
ax.set_ylabel('Time (ms)')
ax.set_title('Camera-to-FPGA processing time per frame')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


# Automatically interpret the measurements from THIS execution.
from IPython.display import display, Markdown

n_frames = len(total_ms)
mean_capture = float(np.mean(capture_ms))
mean_prep = float(np.mean(preprocess_ms))
mean_fpga = float(np.mean(inference_ms))
mean_total = float(np.mean(total_ms))
measured_fps = n_frames / elapsed_s
nominal_fps = 15.0
capture_pct = mean_capture / mean_total * 100.0
prep_pct = mean_prep / mean_total * 100.0
fpga_pct = mean_fpga / mean_total * 100.0
stages = [
    ('Camera capture / OpenCV', mean_capture),
    ('ARM preprocessing', mean_prep),
    ('FPGA inference call + DMA', mean_fpga),
]
dominant_name, dominant_ms = max(stages, key=lambda item: item[1])
vehicle_speed_kmh = 130.0
distance_per_frame_m = vehicle_speed_kmh / 3.6 / measured_fps
distance_during_fpga_m = vehicle_speed_kmh / 3.6 * mean_fpga / 1000.0

report = """
### Experimental results and interpretation

**Measured processing throughput:** **{fps:.2f} frames/s** ({n} consecutive frames).
**Mean complete processing time:** **{total:.2f} ms/frame**.

| Processing stage | Mean time | Share of total |
|---|---:|---:|
| USB camera capture / OpenCV | {capture:.2f} ms | {cp:.1f}% |
| ARM preprocessing | {prep:.2f} ms | {pp:.1f}% |
| FPGA inference call + AXI DMA | {fpga:.2f} ms | {fp:.1f}% |
| **Total** | **{total:.2f} ms** | **100%** |

The largest measured contribution is **{dominant}** ({dominant_time:.2f} ms per frame).
The FPGA figure includes Python buffer operations and DMA, so it is **not the
isolated neural-network core latency**.

### Is this real-time processing?

The measured system throughput is **{fps:.2f} FPS**, compared with the camera's
configured nominal **{nominal:.0f} FPS**. The pipeline successfully processes
consecutive frames, but real-time suitability depends on the application's
required deadline and the actual end-to-end latency.

At **{speed:.0f} km/h**, a vehicle travels about **{distance:.2f} m**
between processed frames at this measured throughput, and **{distance_fpga:.2f} m**
during the mean FPGA inference call. This describes sampling distance, **not**
a validated safe vehicle speed.

### Interpretation of the timing graph

The capture stage may fluctuate because of USB delivery, MJPEG decoding,
exposure and frame synchronization. The first measured frame can take longer
than later frames. The relatively stable FPGA call time is consistent with
fixed-size input and a fixed inference workload.

### Limitations

- The benchmark uses center cropping, not automatic sign localization.
- It reports processing throughput, not full motion-to-result latency or
  road-sign recognition accuracy.
- Camera initialization and warm-up are excluded from the per-frame timings.
- The result applies to the current camera, settings, hardware and scene.
""".format(
    fps=measured_fps, n=n_frames, total=mean_total,
    capture=mean_capture, cp=capture_pct,
    prep=mean_prep, pp=prep_pct,
    fpga=mean_fpga, fp=fpga_pct,
    dominant=dominant_name, dominant_time=dominant_ms,
    nominal=nominal_fps, speed=vehicle_speed_kmh,
    distance=distance_per_frame_m, distance_fpga=distance_during_fpga_m
)
display(Markdown(report))


## Interpretation and questions

1. Check that DMA returns 44 signed `int16` scores. Why must score index 43 be excluded from the traffic-sign classes?
2. Compare the prediction for the labeled test image with its ground-truth class. Is the classification correct?
3. How many images are classified correctly in the 20-image sample? What is the measured mean inference time?
4. Which operations contribute to the measured end-to-end time (Python preprocessing, buffer operations, DMA transfers and FPGA execution)? Does this measurement represent the latency of the neural-network hardware alone?
5. Compare the ARM-based experiment in Lab08.2 with the FPGA experiment here. The two notebooks use **different BNN architectures and timing methods**; explain why their execution times cannot be used to calculate a valid same-model FPGA speedup.
6. Why must the matching `.bit` and `.hwh` files be kept together?
7. How do image scale, camera angle and screen reflections affect camera classification?
8. Based on Section 8, what is the measured complete-frame throughput, and which stage dominates the time? Why is this not the same as the FPGA's isolated hardware latency?
